# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

### Importar

In [2]:
import os
import time
import requests
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
import re


### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

<!-- # Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000` -->

In [5]:
def paso1_tfidf_turismo(ruta_archivo):
    print("--- PASO 1: TF-IDF Corpus Turismo ---")
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        lineas = f.readlines()
    
    corpus_turismo = [re.sub(r'\\s*', '', linea).strip() for linea in lineas if linea.strip()]
    
    vectorizer_turismo = TfidfVectorizer()
    matriz_tfidf_turismo = vectorizer_turismo.fit_transform(corpus_turismo)
    
    print(f"Corpus Turismo procesado: {len(corpus_turismo)} documentos.")
    print(f"Dimensiones de la matriz TF-IDF: {matriz_tfidf_turismo.shape}")
    print(f"Vocabulario extraído: {len(vectorizer_turismo.get_feature_names_out())} términos.\n")
    
    return vectorizer_turismo, matriz_tfidf_turismo

### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [6]:
import requests
import time
from pathlib import Path

def obtener_textos_gutenberg(meta_libros=1000, dir_destino="data/gutenberg_1000"):
    print("--- PASO 2: Construyendo Corpus Gutenberg ---")
    
    ruta = Path(dir_destino)
    ruta.mkdir(parents=True, exist_ok=True)
    
    cantidad_actual = len(list(ruta.glob("*")))
    if cantidad_actual >= meta_libros:
        print(f"Ya existen {cantidad_actual} libros en {dir_destino}. Se omite la descarga.\n")
        return
        
    print(f"Descargando {meta_libros} libros (esto tomará algo de tiempo)...")
    
    contador_exito = 0
    indice_actual = 1
    
    while contador_exito < meta_libros:
        destino_archivo = ruta / f"libro_{indice_actual}.txt"
        
        if destino_archivo.is_file():
            contador_exito += 1
            indice_actual += 1
            continue
            
        direccion_web = f"https://www.gutenberg.org/cache/epub/{indice_actual}/pg{indice_actual}.txt"
        
        try:
            req = requests.get(direccion_web, timeout=5)
            
            if req.status_code == 200 and "text/plain" in req.headers.get("Content-Type", ""):
                destino_archivo.write_text(req.text, encoding='utf-8')
                contador_exito += 1
                
                if contador_exito % 50 == 0:
                    print(f"Descargados {contador_exito}/{meta_libros} libros...")
                    
            time.sleep(0.5)
            
        except requests.RequestException:
            pass
            
        indice_actual += 1
        
    print("Descarga del corpus Gutenberg completada.\n")

### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [7]:
def calcular_tfidf_textos(dir_fuente="data/gutenberg_1000"):
    print("--- PASO 3: TF-IDF Corpus Gutenberg ---")
    
    ruta_base = Path(dir_fuente)
    rutas_txt = list(ruta_base.glob('*.txt'))
    
    coleccion_textos = []
    identificadores = []
    
    print("Cargando libros en memoria...")
    
    for ruta_archivo in rutas_txt:
        contenido = ruta_archivo.read_text(encoding='utf-8', errors='ignore')
        coleccion_textos.append(contenido)
        identificadores.append(ruta_archivo.name)
        
    vectorizador = TfidfVectorizer(stop_words='english', max_features=10000)
    matriz_caracteristicas = vectorizador.fit_transform(coleccion_textos)
    
    print(f"Corpus Gutenberg procesado: {len(coleccion_textos)} documentos.")
    print(f"Dimensiones de la matriz TF-IDF: {matriz_caracteristicas.shape}\n")
    
    return vectorizador, matriz_caracteristicas, identificadores

### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [8]:
def ejecutar_busqueda(texto_query, modelo_vectorial, matriz_datos, identificadores, max_resultados=5):
    query_transformada = modelo_vectorial.transform([texto_query])
    
    puntajes = cosine_similarity(query_transformada, matriz_datos).flatten()
    
    mejores_coincidencias = puntajes.argsort()[-max_resultados:][::-1]
    
    print(f"\n--- Resultados de búsqueda para: '{texto_query}' ---")
    
    for indice in mejores_coincidencias:
        score = puntajes[indice]
        
        if score > 0:
            print(f"Documento: {identificadores[indice]} | Similitud: {score:.4f}")
        else:
            print("No se encontraron más coincidencias relevantes (Similitud = 0).")
            break

### Ejecucion

In [11]:
if __name__ == "__main__":
    import os
    
    dir_actual = os.getcwd()
    if os.path.basename(dir_actual) != 'work' and os.path.exists('work'):
        os.chdir('work')
        dir_actual = os.getcwd()
        
    archivo_turismo = os.path.join(dir_actual, "data", "01_corpus_turismo_500.txt")
    
    if not os.path.exists(archivo_turismo):
        print(f"Error: No se encontró el archivo '{archivo_turismo}'.")
    else:
        print("Archivo encontrado. Iniciando proceso...")
        
        vec_tur, mat_tur = paso1_tfidf_turismo(archivo_turismo)
        
        obtener_textos_gutenberg(meta_libros=1000)
        
        vectorizador_g, matriz_g, identificadores_g = calcular_tfidf_textos()
        
        terminos_consulta = "science fiction adventure"
        ejecutar_busqueda(terminos_consulta, vectorizador_g, matriz_g, identificadores_g)

Archivo encontrado. Iniciando proceso...
--- PASO 1: TF-IDF Corpus Turismo ---
Corpus Turismo procesado: 500 documentos.
Dimensiones de la matriz TF-IDF: (500, 116)
Vocabulario extraído: 116 términos.

--- PASO 2: Construyendo Corpus Gutenberg ---
Descargando 1000 libros (esto tomará algo de tiempo)...
Descargados 50/1000 libros...
Descargados 100/1000 libros...
Descargados 150/1000 libros...
Descargados 200/1000 libros...
Descargados 250/1000 libros...
Descargados 300/1000 libros...
Descargados 350/1000 libros...
Descargados 400/1000 libros...
Descargados 450/1000 libros...
Descargados 500/1000 libros...
Descargados 550/1000 libros...
Descargados 600/1000 libros...
Descargados 650/1000 libros...
Descargados 700/1000 libros...
Descargados 750/1000 libros...
Descargados 800/1000 libros...
Descargados 850/1000 libros...
Descargados 900/1000 libros...
Descargados 950/1000 libros...
Descargados 1000/1000 libros...
Descarga del corpus Gutenberg completada.

--- PASO 3: TF-IDF Corpus Gutenbe